square lattice creation

In [104]:
import numpy as np
from fontTools.diff.color import reset

N = 2 # size of the square
d = 2 # dimensions

# The "All-Up" Lattice
# np.ones creates a grid filled with 1s. We specify 'int' so they are whole numbers.
lattice_up = np.ones((N, N), dtype=int)
print("All-Up Lattice:")
print(lattice_up)

# The Checkerboard Lattice
# We can create this by manually defining the rows as a NumPy array
lattice_checker = np.array([[1, -1],
                            [-1, 1]])
print("\nCheckerboard Lattice:")
print(lattice_checker)

# A Random Lattice
# np.random.choice randomly picks from the list [-1, 1] to fill an NxN grid
lattice_random = np.random.choice([-1, 1], size=(N, N))
print("\nRandom Lattice:")
print(lattice_random)

All-Up Lattice:
[[1 1]
 [1 1]]

Checkerboard Lattice:
[[ 1 -1]
 [-1  1]]

Random Lattice:
[[ 1 -1]
 [ 1 -1]]


periodic neighbour lookup

In [105]:
import random
starting_pos_x = random.randint(0, N - 1)
starting_pos_y = random.randint(0, N - 1)

print("Starting position x: ", starting_pos_x)
print("Starting position y: ", starting_pos_y)
print("")


def get_neighbours(x, y, N, d):
    # Returns the periodic coordinates of the 4 neighbors of atom (x, y)
    top = (x, (y + 1) % d )
    bottom = (x, (y - 1) % d )
    left = ((x - 1) % N, y)
    right = ((x + 1) % N, y)

    # Returning them as a list of coordinate pairs (tuples)
    return [top, bottom, left, right]

# Now you can instantly look up neighbors for ANY atom:
my_neighbours = get_neighbours(starting_pos_x, starting_pos_y, N, d)
print(my_neighbours)

Starting position x:  0
Starting position y:  1

[(0, 0), (0, 0), (1, 1), (1, 1)]


## Total Energy
$E = -J \sum_{\langle ij \rangle} s_i s_j - B\sum_{\langle i \rangle} s_i$

In [106]:
J = 1
total_energy = 0

# 1. Loop through coordinates
for y in range(N):
    for x in range(N):
        current_spin = lattice_up[y, x]

        # 2. Get the neighbors (using the logic you already wrote)
        neighbours = get_neighbours(x, y, N, d)

        # 3. Multiply and sum the interactions
        for nx, ny in neighbours:
            neighbour_spin = lattice_up[ny, nx]

            # The +1/-1 math handles the Up/Down states automatically!
            total_energy += -J * (current_spin * neighbour_spin)

# 4. Fix the double-counting
total_energy = total_energy / 2
total_energy_up = total_energy

print("Lattice up")
print("Total Energy:", total_energy)
print("")

total_energy = 0
# 1. Loop through coordinates
for y in range(N):
    for x in range(N):
        current_spin = lattice_checker[y, x]

        # 2. Get the neighbors (using the logic you already wrote)
        neighbours = get_neighbours(x, y, N, d)

        # 3. Multiply and sum the interactions
        for nx, ny in neighbours:
            neighbour_spin = lattice_checker[ny, nx]

            # The +1/-1 math handles the Up/Down states automatically!
            total_energy += -J * (current_spin * neighbour_spin)

# 4. Fix the double-counting
total_energy = total_energy / 2
total_energy_checker = total_energy

print("Lattice checker")
print("Total Energy:", total_energy)
print("")

total_energy = 0
# 1. Loop through coordinates
for y in range(N):
    for x in range(N):
        current_spin = lattice_random[y, x]

        # 2. Get the neighbors (using the logic you already wrote)
        neighbours = get_neighbours(x, y, N, d)

        # 3. Multiply and sum the interactions
        for nx, ny in neighbours:
            neighbour_spin = lattice_random[ny, nx]

            # The +1/-1 math handles the Up/Down states automatically!
            total_energy += -J * (current_spin * neighbour_spin)

# 4. Fix the double-counting
total_energy = total_energy / 2
total_energy_random = total_energy

print("Random Lattice")
print("Total Energy:", total_energy)
print("")

Lattice up
Total Energy: -8.0

Lattice checker
Total Energy: 8.0

Random Lattice
Total Energy: 0.0



## Average magnetisation (m)/ energy per spin
$m = \frac{1}{N} \sum_{\langle i \rangle} s_i$

In [107]:
total_mag = 0
count = 0
for row in lattice_up:
    for col in row:
        total_mag += col
        count += 1

Avg_mag = total_mag / count
Avg_energy = total_energy_up / count
abs_mag = abs(total_mag)
print("Lattice Up")
print("Average magnetisation (m) per spin: ", Avg_mag)
print("Absolute magnetisation (m) per spin: ", abs_mag)
print("Average energy per spin: ", Avg_energy)
print("")

total_mag = 0
count = 0

for row in lattice_checker:
    for col in row:
        total_mag += col
        count += 1

Avg_mag = total_mag / count
Avg_energy = total_energy_checker / count
abs_mag = abs(total_mag)
print("Lattice Checker")
print("Average magnetisation (m) per spin: ", Avg_mag)
print("Absolute magnetisation (m) per spin: ", abs_mag)
print("Average energy per spin: ", Avg_energy)
print("")

total_mag = 0
count = 0

for row in lattice_random:
    for col in row:
        total_mag += col
        count += 1

Avg_mag = total_mag / count
Avg_energy = total_energy_random / count
abs_mag = abs(total_mag)
print("Lattice Random")
print("Average magnetisation (m) per spin: ", Avg_mag)
print("Absolute magnetisation (m) per spin: ", abs_mag)
print("Average energy per spin: ", Avg_energy)
print("")

Lattice Up
Average magnetisation (m) per spin:  1.0
Absolute magnetisation (m) per spin:  4
Average energy per spin:  -2.0

Lattice Checker
Average magnetisation (m) per spin:  0.0
Absolute magnetisation (m) per spin:  0
Average energy per spin:  2.0

Lattice Random
Average magnetisation (m) per spin:  0.0
Absolute magnetisation (m) per spin:  0
Average energy per spin:  0.0



## Local Change in Energy ($\Delta E$)
$\Delta E = 2 J s_i \sum s_{neighbors}$

In [108]:
# Choose ONE atom to test (for example, the top-left corner)
x, y = 0, 0
current_spin = lattice_up[y, x]

# Get its 4 neighbors using your function
neighbours = get_neighbours(x, y, N, d)
sum_neighbours = 0

# Sum up the spins of those 4 neighbors
for nx, ny in neighbours:
    sum_neighbours += lattice_up[ny, nx]

# Calculate how much the total energy WOULD change if we flipped (x,y)
delta_E = 2 * J * current_spin * sum_neighbours
print("Lattice Up")
print("Local Change: ", delta_E)
print("")


# Lattice checker
current_spin = lattice_checker[y, x]
sum_neighbours = 0

# Sum up the spins of those 4 neighbors
for nx, ny in neighbours:
    sum_neighbours += lattice_checker[ny, nx]

delta_E = 2 * J * current_spin * sum_neighbours
print("Lattice Checker")
print("Local Change: ", delta_E)
print("")


# Lattice random
current_spin = lattice_random[y, x]
sum_neighbours = 0

for nx, ny in neighbours:
    sum_neighbours += lattice_random[ny, nx]

# Sum up the spins of those 4 neighbors
delta_E = 2 * J * current_spin * sum_neighbours
print("Random Lattice")
print("Local Change: ", delta_E)
print("")

Lattice Up
Local Change:  8

Lattice Checker
Local Change:  -8

Random Lattice
Local Change:  0

